In [ ]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import cv2
import os
import numpy as np

In [57]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware initialized. Computing on: {device}\n")

Hardware initialized. Computing on: cpu



In [51]:
class PublicVideoDataset(Dataset):
    def __init__(self, root_dir, alpha=4, fast_frames=32, target_size=(224, 224)):
        self.root_dir = root_dir
        self.alpha = alpha
        self.fast_frames = fast_frames
        self.target_size = target_size

        # Dynamically map folder names to numerical classes
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        self.video_paths = []
        self.labels = []

        for cls_name in self.classes:
            cls_dir = os.path.join(root_dir, cls_name)
            if os.path.isdir(cls_dir):
                for file in os.listdir(cls_dir):
                    if file.endswith(('.mp4', '.avi')):
                        self.video_paths.append(os.path.join(cls_dir, file))
                        self.labels.append(self.class_to_idx[cls_name])

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]

        cap = cv2.VideoCapture(video_path)
        processed_frames = []

        for _ in range(self.fast_frames):
            ret, frame = cap.read()
            if ret:
                resized = cv2.resize(frame, self.target_size)
                resized = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
                processed_frames.append(resized)
            else:
                processed_frames.append(np.zeros((self.target_size[1], self.target_size[0], 3), dtype=np.uint8))
        cap.release()

        video_tensor = torch.tensor(np.array(processed_frames), dtype=torch.float32)
        video_tensor = video_tensor.permute(3, 0, 1, 2) / 255.0

        fast_tensor = video_tensor
        slow_tensor = video_tensor[:, ::self.alpha, :, :]

        return [slow_tensor, fast_tensor], label, video_path


In [38]:
# Define dataset paths
train_dir = r"/Action_recognition_slowfast/experiment_1/public_dataset\train"
val_dir = r"/Action_recognition_slowfast/experiment_1/public_dataset\val"

if not os.path.exists(train_dir) or not os.path.exists(val_dir):
    raise FileNotFoundError("CRITICAL: 'public_dataset/train' or 'public_dataset/val' not found. Please ensure your folders are structured correctly next to this notebook.")

train_dataset = PublicVideoDataset(root_dir=train_dir)
val_dataset = PublicVideoDataset(root_dir=val_dir)
action_classes = train_dataset.classes
num_classes = len(action_classes)

print(f"Loaded {num_classes} action categories: {action_classes}")
print(f"Dataset Split: {len(train_dataset)} Training videos, {len(val_dataset)} Validation videos.\n")

train_loader = DataLoader(dataset=train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=4, shuffle=False)

Loaded 3 action categories: ['Red card', 'Scoring', 'Tackling']
Dataset Split: 322 Training videos, 82 Validation videos.



In [39]:
# PHASE 2: ARCHITECTURE ASSEMBLY

model = torch.hub.load('facebookresearch/pytorchvideo', 'slowfast_r50', pretrained=True)

# Transplant the classification head for your specific actions
in_features = model.blocks[6].proj.in_features
model.blocks[6].proj = nn.Linear(in_features=in_features, out_features=num_classes)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
print("Architecture successfully adapted to football dataset.\n")

Using cache found in C:\Users\omoni/.cache\torch\hub\facebookresearch_pytorchvideo_main


Architecture successfully adapted to football dataset.



In [55]:
print(model)

Net(
  (blocks): ModuleList(
    (0): MultiPathWayWithFuse(
      (multipathway_blocks): ModuleList(
        (0): ResNetBasicStem(
          (conv): Conv3d(3, 64, kernel_size=(1, 7, 7), stride=(1, 2, 2), padding=(0, 3, 3), bias=False)
          (norm): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation): ReLU()
          (pool): MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=[0, 1, 1], dilation=1, ceil_mode=False)
        )
        (1): ResNetBasicStem(
          (conv): Conv3d(3, 8, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False)
          (norm): BatchNorm3d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation): ReLU()
          (pool): MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=[0, 1, 1], dilation=1, ceil_mode=False)
        )
      )
      (multipathway_fusion): FuseFastToSlow(
        (conv_fast_to_slow): Conv3d(8, 16, kernel_size=(7, 1, 1), st

In [56]:
# PHASE 3: THE TRAINING ENGINE (With History Tracking)

epochs = 3
best_accuracy = 0.0
weights_path = "best_football_model.pth"

# 1. Create empty lists to act as our "history" object
history_train_loss = []
history_val_accuracy = []

for epoch in range(epochs):
    print(f"--- Epoch {epoch+1}/{epochs} ---")

    # Training Phase
    model.train()
    running_loss = 0.0
    for videos, labels, _ in train_loader:
        slow_inputs, fast_inputs = videos[0].to(device), videos[1].to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model([slow_inputs, fast_inputs])
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Calculate the average loss for this epoch
    epoch_loss = running_loss / len(train_loader)
    print(f"  Training Loss: {epoch_loss:.4f}")

    # 2. Save the loss to our history list
    history_train_loss.append(epoch_loss)

    # Validation Phase
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for videos, labels, _ in val_loader:
            slow_inputs, fast_inputs = videos[0].to(device), videos[1].to(device)
            labels = labels.to(device)

            outputs = model([slow_inputs, fast_inputs])
            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    # Calculate the percentage accuracy for this epoch
    accuracy = (correct / total) * 100
    print(f"  Validation Accuracy: {accuracy:.2f}%")

    # 3. Save the accuracy to our history list
    history_val_accuracy.append(accuracy)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model.state_dict(), weights_path)
        print(" Network learned! Optimal weights saved.")
print("\nTraining Phase Complete.\n")

--- Epoch 1/3 ---
  Training Loss: 0.0456
  Validation Accuracy: 96.34%
  🏆 Network learned! Optimal weights saved.
--- Epoch 2/3 ---
  Training Loss: 0.0433
  Validation Accuracy: 96.34%
--- Epoch 3/3 ---
  Training Loss: 0.0340
  Validation Accuracy: 93.90%

Training Phase Complete.



In [61]:
# PHASE 4: AUTOMATED INFERENCE DEMONSTRATION

# 1. Grab the very first video from the validation set to test
sample_video_path = val_dataset.video_paths[0]
actual_label_idx = val_dataset.labels[0]
actual_action = action_classes[actual_label_idx]

print(f"Loading optimal weights from '{weights_path}'...")
model.load_state_dict(torch.load(weights_path, map_location=device))
model.eval()

print(f"Target Video: {sample_video_path}")
print(f"Ground Truth (Actual Action): {actual_action}")
print("Analyzing spatial and temporal biomechanics...")

# 2. Process the single clip
cap = cv2.VideoCapture(sample_video_path)
frames = []
for _ in range(32):
    ret, frame = cap.read()
    if ret:
        resized = cv2.cvtColor(cv2.resize(frame, (224, 224)), cv2.COLOR_BGR2RGB)
        frames.append(resized)
    else:
        frames.append(np.zeros((224, 224, 3), dtype=np.uint8))
cap.release()

video_tensor = torch.tensor(np.array(frames), dtype=torch.float32).permute(3, 0, 1, 2) / 255.0
video_tensor = video_tensor.unsqueeze(0) # Add batch dimension

fast_tensor = video_tensor.to(device)
slow_tensor = video_tensor[:, :, ::4, :, :].to(device)

# 3. Make the prediction
with torch.no_grad():
    outputs = model([slow_tensor, fast_tensor])
    probabilities = torch.nn.functional.softmax(outputs, dim=1)
    confidence, predicted_idx = torch.max(probabilities, 1)

predicted_action = action_classes[predicted_idx.item()]
confidence_score = confidence.item() * 100

print("FINAL PREDICTION RESULT:")
print(f"Predicted Action : {predicted_action}")
print(f"Confidence Score : {confidence_score:.2f}%")
if predicted_action == actual_action:
    print("SUCCESS")
else:
    print("FAILED")

Loading optimal weights from 'best_football_model.pth'...
Target Video: C:\Users\omoni\Desktop\experiments\Action_recognition_slowfast\public_dataset\val\Red card\Red Card 50.avi
Ground Truth (Actual Action): Red card
Analyzing spatial and temporal biomechanics...
FINAL PREDICTION RESULT:
Predicted Action : Red card
Confidence Score : 100.00%
SUCCESS
